# CrewAI Real Estate Analysis Crew

**Project:** Multi-Agent Property Investment Analysis Pipeline  
**Framework:** CrewAI (multi-agent orchestration)  
**LLM Backend:** Ollama (local, qwen3 model)  

---

## Overview

This notebook builds a **4-agent real estate analysis crew** that evaluates properties for investment potential — from market fundamentals through neighborhood scoring, pricing strategy, and risk assessment.

### The Agent Roster

| # | Agent | Role | Deliverable |
|---|-------|------|------------|
| 1 | **Market Analyst** | Macro & Micro Market Intelligence | Market conditions report — supply/demand, price trends, economic drivers |
| 2 | **Neighborhood Scorer** | Location Quality Assessor | Neighborhood scorecard — amenities, safety, schools, walkability, growth trajectory |
| 3 | **Pricing Agent** | Valuation & Deal Structuring | Pricing analysis — fair market value, comparable sales, offer strategy |
| 4 | **Risk Reviewer** | Investment Risk Assessor | Risk report — downside scenarios, regulatory risks, exit strategy viability |

### Multi-Agent Property Analysis Flow

```
Property Details + Location
          │
          ▼
┌─────────────────────┐
│   Market Analyst    │  "WHAT's the market doing?" — macro trends, supply/demand, forecast
└──────────┬──────────┘
           │ Market conditions report
           ▼
┌─────────────────────┐
│ Neighborhood Scorer │  "WHERE exactly?" — location quality, amenities, growth potential
└──────────┬──────────┘
           │ Neighborhood scorecard with ratings
           ▼
┌─────────────────────┐
│   Pricing Agent     │  "HOW MUCH?" — valuation, comps analysis, offer strategy
└──────────┬──────────┘
           │ Pricing analysis with offer range
           ▼
┌─────────────────────┐
│   Risk Reviewer     │  "WHAT COULD GO WRONG?" — downside scenarios, deal-breakers
└─────────────────────┘
           │
           ▼
  Investment decision package
```

### Why Multi-Agent for Real Estate?

Real estate investment analysis requires fundamentally different analytical modes:

1. **Market analysis is macro** — it looks at trends, economics, and aggregate data
2. **Neighborhood scoring is micro** — it evaluates the specific block, street, and surrounding amenities
3. **Pricing is comparative** — it requires disciplined comp analysis against similar sold properties
4. **Risk assessment is adversarial** — the risk reviewer's job is to find reasons NOT to invest

Combining all four in a single prompt produces a biased result — the LLM tends to either be too bullish or too bearish. Separating them into specialized agents with distinct cognitive modes produces balanced, actionable analysis.

## 1. Environment Setup

In [ ]:
# Install dependencies
!pip install -q crewai crewai-tools langchain-ollama

In [ ]:
import os
import json
import textwrap
from datetime import datetime

from crewai import Agent, Task, Crew, Process
from crewai import LLM

print("CrewAI imports successful")
print(f"Timestamp: {datetime.now().isoformat()}")

## 2. LLM Configuration

Every agent uses the same local Ollama model. The differentiation comes entirely from role design — backstory, goal, and task description — not model selection.

In [ ]:
llm = LLM(
    model="ollama/qwen3",
    base_url="http://localhost:11434",
    temperature=0.7,
)

print(f"LLM configured: {llm.model}")

## 3. Define the Agents

### Structuring a Domain-Expert Crew

Real estate professionals work in specialized roles. Our crew mirrors this:

| Real-World Role | CrewAI Agent | Analytical Mode |
|----------------|-------------|----------------|
| Market Research Analyst | Market Analyst | **Contextual** — interprets macro data and local trends |
| Local Area Expert / Realtor | Neighborhood Scorer | **Observational** — evaluates tangible location attributes |
| Appraiser / Deal Analyst | Pricing Agent | **Quantitative** — compares numbers, calculates value |
| Due Diligence / Risk Manager | Risk Reviewer | **Adversarial** — stress-tests assumptions, finds flaws |

**Structural insight:** The agents progress from broad (market conditions) to narrow (specific property risk). Each layer adds specificity and accumulates context from the previous agents.

### 3.1 Market Analyst Agent

**Pipeline role:** Sets the macro context. Every downstream agent's analysis is shaped by whether the market is hot, cooling, or distressed.

**Design rationale:** The backstory emphasizes both quantitative metrics (inventory months, price-to-income) and qualitative signals (employer expansions, infrastructure projects). This dual lens prevents the agent from producing a purely statistical report that misses real-world catalysts.

In [ ]:
market_analyst = Agent(
    role="Real Estate Market Intelligence Analyst",
    goal=(
        "Analyze the macro and micro market conditions for the specified property "
        "location. Assess supply/demand dynamics, price trends (1-year, 3-year, 5-year), "
        "economic drivers, employment trends, population growth, and forecast the market "
        "direction for the next 12-24 months."
    ),
    backstory=(
        "You are a real estate market analyst with 14 years of experience at a top "
        "investment advisory firm. You have analyzed 200+ markets across the US and "
        "developed a proprietary framework for market assessment that combines: "
        "(1) Supply metrics — active inventory, months of supply, new construction permits, "
        "absorption rate, (2) Demand signals — population growth, job creation, wage growth, "
        "migration patterns, mortgage rate sensitivity, (3) Price dynamics — median price "
        "trends, price-to-income ratio, price-to-rent ratio, list-to-sale ratio, days on "
        "market trends, (4) Economic catalysts — major employer expansions, infrastructure "
        "projects, university/hospital presence, tech hub formation. You classify markets "
        "into phases: Recovery, Expansion, Hyper-Supply, and Recession. Your reports always "
        "include a 12-month directional forecast with confidence level and key risk factors "
        "that could change the trajectory."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print(f"Agent created: {market_analyst.role}")

### 3.2 Neighborhood Scorer Agent

**Pipeline role:** Zooms from the macro market down to the specific location. A strong market can have weak neighborhoods, and vice versa — this agent captures that nuance.

**Design rationale:** The backstory defines a structured 10-category scorecard rather than asking for a general description. This produces consistent, comparable output across different properties.

In [ ]:
neighborhood_scorer = Agent(
    role="Neighborhood Quality & Growth Potential Assessor",
    goal=(
        "Evaluate the specific neighborhood of the property across 10 quality "
        "dimensions. Produce a structured scorecard with ratings (1-10) for each "
        "dimension and an overall neighborhood grade (A through F). Identify the "
        "top 3 neighborhood strengths and the top 3 concerns."
    ),
    backstory=(
        "You are an urban planning consultant and neighborhood evaluation specialist "
        "who has scored 500+ neighborhoods for real estate investment funds. You use a "
        "structured 10-dimension scorecard: (1) Safety & Crime — crime rate trends, "
        "police response, community watch, (2) Schools — K-12 ratings, proximity, school "
        "choice options, (3) Walkability & Transit — walk score, bike score, public transit "
        "access, (4) Amenities — restaurants, grocery stores, parks, gyms, healthcare, "
        "(5) Employment Centers — commute time to major employers, job density, "
        "(6) Demographics — median income, education level, age distribution, diversity, "
        "(7) Property Values Trend — neighborhood-level appreciation vs. city average, "
        "(8) Development Pipeline — planned projects, zoning changes, gentrification signals, "
        "(9) Environmental Quality — noise, air quality, flood risk, green space, "
        "(10) Community Character — resident tenure, HOA presence, local events, pride of "
        "ownership. Each dimension gets a 1-10 score with a brief evidence note. You also "
        "identify the neighborhood phase: Emerging, Established, Premium, or Declining."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print(f"Agent created: {neighborhood_scorer.role}")

### 3.3 Pricing Agent

**Pipeline role:** Translates qualitative analysis into numbers. Uses the market context and neighborhood score to determine fair value and construct an offer strategy.

**Design rationale:** The backstory mandates a specific valuation methodology (comparable sales approach with adjustments) rather than allowing the agent to just guess a price. The emphasis on adjustment factors forces systematic, defensible valuations.

In [ ]:
pricing_agent = Agent(
    role="Property Valuation & Deal Structuring Specialist",
    goal=(
        "Determine the fair market value of the property using comparable sales "
        "analysis. Provide a pricing range (low, mid, high), identify the best "
        "offer strategy, calculate key investment metrics (cap rate, cash-on-cash "
        "return, GRM), and recommend a specific offer price with justification."
    ),
    backstory=(
        "You are a certified real estate appraiser (MAI designation) and deal analyst "
        "with 10 years at a real estate private equity firm. Your valuation approach is "
        "rigorous and defensible: (1) Comparable Sales Analysis — identify 4-6 recently "
        "sold properties with similar characteristics, then apply adjustment factors for "
        "differences in size, condition, lot size, age, and features, (2) Income Approach "
        "— for investment properties, estimate gross rent, vacancy rate, operating expenses, "
        "and NOI to derive value via cap rate, (3) Replacement Cost — what would it cost to "
        "build this property today? (4) Market Sentiment Adjustment — in hot markets, buyers "
        "pay a premium; in cold markets, discounts are available. Your pricing outputs always "
        "include: estimated fair market value (range), recommended offer price, maximum walk-away "
        "price, and a deal quality grade (A through D). You never recommend paying above fair "
        "value unless the investment thesis has a specific catalyst (e.g., rezoning, value-add "
        "renovation) that justifies the premium."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print(f"Agent created: {pricing_agent.role}")

### 3.4 Risk Reviewer Agent

**Pipeline role:** The adversarial check. This agent's explicit job is to find reasons the deal could go wrong — counterbalancing the natural optimism bias of the upstream agents.

**Design rationale:** The backstory defines a structured risk framework with specific categories and a severity matrix. Without this structure, risk assessments tend to be vague warnings ("the market could decline") rather than specific, actionable risk factors with mitigation strategies.

In [ ]:
risk_reviewer = Agent(
    role="Investment Risk Analyst & Due Diligence Specialist",
    goal=(
        "Conduct a comprehensive risk assessment of the property investment. "
        "Identify all material risks across 6 categories, rate each risk by "
        "likelihood and impact, propose mitigations, and deliver a GO / CAUTION / "
        "NO-GO recommendation with specific conditions."
    ),
    backstory=(
        "You are a real estate risk management specialist who has reviewed 300+ "
        "investment deals for a family office managing $2B in real estate assets. "
        "Your role is explicitly adversarial: your job is to find what could go wrong, "
        "not to validate the investment thesis. You evaluate risk across 6 categories: "
        "(1) Market Risk — what if the market turns? How correlated is this property to "
        "macro cycles? (2) Property-Specific Risk — structural issues, deferred maintenance, "
        "environmental contamination, title defects, (3) Regulatory Risk — zoning changes, "
        "rent control, property tax increases, building code compliance, (4) Financial Risk — "
        "interest rate exposure, vacancy risk, tenant concentration, insurance costs, "
        "(5) Liquidity Risk — how easy is it to sell this property? Days on market for comps, "
        "buyer pool depth, (6) Opportunity Cost — is the capital better deployed elsewhere? "
        "For each risk, you assign a Likelihood (Low/Medium/High) and Impact (Low/Medium/High) "
        "rating, forming a risk matrix. You always include a Exit Strategy Assessment: could "
        "the investor sell this in 3 years at a profit? In a down market? You deliver a final "
        "recommendation: GO (proceed with confidence), CAUTION (proceed with specific conditions), "
        "or NO-GO (deal-breakers present)."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print(f"Agent created: {risk_reviewer.role}")

### Agent Roster Summary

In [ ]:
agents = [market_analyst, neighborhood_scorer, pricing_agent, risk_reviewer]

print("=" * 70)
print("REAL ESTATE ANALYSIS CREW ROSTER")
print("=" * 70)
for i, agent in enumerate(agents, 1):
    print(f"\n[{i}] {agent.role}")
    print(f"    Goal: {agent.goal[:90]}...")
    print(f"    Delegation: {'Enabled' if agent.allow_delegation else 'Disabled'}")
print("\n" + "=" * 70)

## 4. Define Tasks & Property Pipeline

### Pipeline Architecture: Broad-to-Narrow Analysis

The task flow mirrors professional real estate due diligence:

| Phase | Agent | Scope | Question |
|-------|-------|-------|----------|
| **Context** | Market Analyst | Metro / city level | Is this a good market to invest in right now? |
| **Location** | Neighborhood Scorer | Block / street level | Is this a good location within the market? |
| **Value** | Pricing Agent | Property level | Is this property priced fairly for the location? |
| **Risk** | Risk Reviewer | Deal level | What could go wrong with this specific investment? |

**Key structural insight:** Each phase narrows the geographic and analytical scope while accumulating context from broader analyses. The risk reviewer sees everything — market conditions, neighborhood quality, AND pricing analysis — giving it the full picture to assess deal viability.

**Context accumulation:** In CrewAI's sequential process, each agent automatically sees all prior agents' outputs:
- Neighborhood Scorer sees: market conditions report
- Pricing Agent sees: market conditions + neighborhood scorecard
- Risk Reviewer sees: market conditions + neighborhood + pricing analysis

### 4.1 Configure the Property

The pipeline is parameterized — change the property details and re-run to analyze any investment opportunity.

In [ ]:
# =====================================================
# CONFIGURE YOUR PROPERTY HERE
# =====================================================
PROPERTY = {
    "address": "742 Evergreen Terrace, Austin, TX 78704",
    "type": "Single-Family Home",
    "bedrooms": 3,
    "bathrooms": 2,
    "sqft": 1_850,
    "lot_sqft": 6_500,
    "year_built": 2005,
    "asking_price": 485_000,
    "condition": "Good — updated kitchen, original bathrooms, new roof (2022)",
    "intent": "Buy-and-hold rental investment",
    "estimated_rent": 2_400,
}
# =====================================================

print("PROPERTY DETAILS")
print("=" * 50)
for k, v in PROPERTY.items():
    label = k.replace("_", " ").title()
    if isinstance(v, int) and v > 1000:
        print(f"  {label:<20} {v:>,}")
    else:
        print(f"  {label:<20} {v}")

### 4.2 Task 1 — Market Analysis

**Pipeline phase:** Context setting — establishes the macro conditions that all downstream analysis depends on.

**Handoff:** The market conditions report shapes the neighborhood scorer's expectations and the pricing agent's market adjustment factors.

In [ ]:
prop_summary = (
    f"Property: {PROPERTY['address']}\n"
    f"Type: {PROPERTY['type']} | {PROPERTY['bedrooms']}bd/{PROPERTY['bathrooms']}ba | "
    f"{PROPERTY['sqft']:,} sqft on {PROPERTY['lot_sqft']:,} sqft lot\n"
    f"Year Built: {PROPERTY['year_built']} | Condition: {PROPERTY['condition']}\n"
    f"Asking Price: ${PROPERTY['asking_price']:,} | Est. Rent: ${PROPERTY['estimated_rent']:,}/mo\n"
    f"Intent: {PROPERTY['intent']}"
)

task_market = Task(
    description=(
        f"Analyze the real estate market conditions for this property:\n\n"
        f"{prop_summary}\n\n"
        "Your market report MUST cover:\n"
        "1. **Market Phase**: Recovery / Expansion / Hyper-Supply / Recession — with evidence\n"
        "2. **Supply Metrics**: Estimated months of inventory, new construction activity, "
        "   absorption rate\n"
        "3. **Demand Drivers**: Population growth, employment trends, major employers, "
        "   in-migration patterns for this metro area\n"
        "4. **Price Trends**: Median price movement (1yr, 3yr, 5yr), price-to-income ratio, "
        "   price-to-rent ratio\n"
        "5. **Interest Rate Impact**: How current mortgage rates affect buyer demand and "
        "   affordability in this market\n"
        "6. **12-Month Forecast**: Market direction prediction with confidence level (Low/"
        "   Medium/High) and key variables that could change the trajectory\n"
        "7. **Investment Climate Grade**: A-F rating for buy-and-hold rental investment\n"
    ),
    expected_output=(
        "A market conditions report covering market phase, supply/demand dynamics, "
        "price trends, interest rate impact, 12-month forecast with confidence level, "
        "and an investment climate grade."
    ),
    agent=market_analyst,
)

print(f"Task created: Market Analysis -> {task_market.agent.role}")

### 4.3 Task 2 — Neighborhood Scoring

**Pipeline phase:** Location evaluation — drills down from the city-wide market analysis to the specific neighborhood.

**Handoff:** The neighborhood scorecard directly influences the pricing agent's location adjustment factor and the risk reviewer's location-related risks.

In [ ]:
task_neighborhood = Task(
    description=(
        f"Score the neighborhood for this property:\n\n"
        f"{prop_summary}\n\n"
        "Evaluate the neighborhood across 10 dimensions (rate each 1-10):\n"
        "1. **Safety & Crime** — crime statistics, trends, community safety perception\n"
        "2. **Schools** — K-12 quality ratings, proximity, magnet/charter options\n"
        "3. **Walkability & Transit** — walk score estimate, bike infrastructure, bus/rail access\n"
        "4. **Amenities** — dining, shopping, grocery, parks, recreation, healthcare proximity\n"
        "5. **Employment Access** — commute to major employment centers, remote-work friendliness\n"
        "6. **Demographics** — median household income, education levels, age mix\n"
        "7. **Property Value Trend** — neighborhood-level appreciation vs. city-wide average\n"
        "8. **Development Pipeline** — planned construction, zoning changes, gentrification signals\n"
        "9. **Environmental Quality** — flood risk, noise levels, air quality, green space\n"
        "10. **Community Character** — resident tenure, neighborhood pride, local events\n\n"
        "Produce:\n"
        "- A scorecard table with all 10 dimensions\n"
        "- Overall neighborhood grade (A through F)\n"
        "- Neighborhood phase: Emerging / Established / Premium / Declining\n"
        "- Top 3 strengths with evidence\n"
        "- Top 3 concerns with evidence\n"
        "- Rental demand assessment: How easy will it be to find quality tenants here?\n"
    ),
    expected_output=(
        "A neighborhood scorecard with 10 dimension ratings (1-10), overall grade, "
        "neighborhood phase, top strengths and concerns, and rental demand assessment."
    ),
    agent=neighborhood_scorer,
)

print(f"Task created: Neighborhood Scoring -> {task_neighborhood.agent.role}")

### 4.4 Task 3 — Pricing Analysis

**Pipeline phase:** Valuation — transforms the qualitative market and neighborhood data into a specific dollar figure with range and justification.

**Handoff:** The pricing analysis gives the risk reviewer concrete numbers to stress-test and an offer strategy to evaluate.

In [ ]:
task_pricing = Task(
    description=(
        f"Determine fair market value and offer strategy for this property:\n\n"
        f"{prop_summary}\n\n"
        "Using the market conditions report and neighborhood scorecard from previous "
        "agents, produce:\n\n"
        "1. **Comparable Sales Analysis**: Identify 4-6 hypothetical comparable properties "
        "   that recently sold, with adjustment factors for differences\n"
        "2. **Fair Market Value Range**: Low / Mid / High estimates with methodology\n"
        "3. **Income Valuation** (for rental intent):\n"
        "   - Gross Rent Multiplier (GRM)\n"
        "   - Net Operating Income (NOI) estimate\n"
        "   - Cap Rate analysis (property vs. market average)\n"
        "   - Cash-on-cash return estimate (assuming 25% down, 7% rate, 30yr)\n"
        "4. **Asking Price Assessment**: Is the asking price fair, high, or low vs. your "
        "   estimated FMV? By how much?\n"
        "5. **Offer Strategy**:\n"
        "   - Recommended initial offer\n"
        "   - Maximum price (walk-away threshold)\n"
        "   - Negotiation leverage points\n"
        "   - Suggested contingencies\n"
        "6. **Deal Quality Grade**: A (strong buy) / B (fair deal) / C (marginal) / D (overpriced)\n"
    ),
    expected_output=(
        "A pricing analysis with comparable sales, fair market value range, income "
        "valuation metrics (GRM, NOI, cap rate, CoC return), offer strategy with "
        "recommended price ranges, and a deal quality grade."
    ),
    agent=pricing_agent,
)

print(f"Task created: Pricing Analysis -> {task_pricing.agent.role}")

### 4.5 Task 4 — Risk Review

**Pipeline phase:** Adversarial assessment — stress-tests the entire investment thesis built by the first three agents.

**Handoff:** Final deliverable — the risk assessment incorporates and challenges all prior analyses to produce a GO / CAUTION / NO-GO recommendation.

In [ ]:
task_risk = Task(
    description=(
        f"Conduct a comprehensive risk assessment for this property investment:\n\n"
        f"{prop_summary}\n\n"
        "Using ALL prior analyses (market conditions, neighborhood scorecard, pricing), "
        "evaluate risk across 6 categories:\n\n"
        "1. **Market Risk** — sensitivity to market cycles, correlation with broader economy, "
        "   what happens if the market declines 10-20%?\n"
        "2. **Property-Specific Risk** — condition concerns, age-related issues, capex "
        "   requirements in next 5 years, insurance considerations\n"
        "3. **Regulatory Risk** — property tax trajectory, potential rent control, zoning "
        "   changes, landlord-tenant law environment\n"
        "4. **Financial Risk** — interest rate exposure, vacancy assumptions, rental income "
        "   variability, operating expense inflation\n"
        "5. **Liquidity Risk** — estimated time to sell, buyer pool depth, market conditions "
        "   impact on exit options\n"
        "6. **Opportunity Cost** — could this capital earn more elsewhere? Compare to "
        "   alternative investments (REITs, index funds, other markets)\n\n"
        "For EACH risk, provide:\n"
        "- Description of the specific risk\n"
        "- Likelihood: Low / Medium / High\n"
        "- Impact: Low / Medium / High\n"
        "- Mitigation strategy\n\n"
        "Also include:\n"
        "- **Stress Test**: What happens to returns if (a) vacancy doubles, (b) rates rise "
        "  2%, (c) property value drops 15%?\n"
        "- **Exit Strategy Assessment**: Can the investor sell profitably in 3 years? In 5?\n"
        "- **Deal-Breakers**: Are there any absolute NO-GO factors?\n"
        "- **Final Recommendation**: GO / CAUTION / NO-GO with specific conditions\n"
    ),
    expected_output=(
        "A risk assessment report with 6-category risk analysis (likelihood/impact matrix), "
        "stress test results, exit strategy assessment, deal-breaker check, and a "
        "GO/CAUTION/NO-GO recommendation."
    ),
    agent=risk_reviewer,
)

print(f"Task created: Risk Review -> {task_risk.agent.role}")

### Pipeline Flow Visualization

In [ ]:
tasks = [task_market, task_neighborhood, task_pricing, task_risk]
task_names = ["Market Analysis", "Neighborhood Scoring", "Pricing Analysis", "Risk Review"]
flow_labels = [
    "CONTEXT — market conditions & investment climate",
    "LOCATION — neighborhood scorecard with 10-dim ratings",
    "VALUE — fair market value, comps, offer strategy",
    "RISK — adversarial review, stress test, GO/CAUTION/NO-GO",
]

print("PROPERTY ANALYSIS PIPELINE")
print("=" * 70)
for i, (name, task, flow) in enumerate(zip(task_names, tasks, flow_labels), 1):
    print(f"  Phase {i}: {name}")
    print(f"    Agent: {task.agent.role}")
    print(f"    Output: {flow}")
    if i < len(tasks):
        print(f"    {'':8}│")
        print(f"    {'':8}│  (context accumulates ↓)")
        print(f"    {'':8}│")
print("=" * 70)
print(f"\nRisk Reviewer sees ALL outputs from Phases 1 + 2 + 3")

## 5. Assemble and Run the Crew

### Sequential Process for Due Diligence

Sequential execution is the right choice for property analysis because:

| Reason | Explanation |
|--------|------------|
| **Dependency chain** | You can't price a property without knowing the market conditions |
| **Context accumulation** | The risk reviewer needs the complete picture |
| **Natural workflow** | Mirrors how professional due diligence actually works |
| **Quality control** | Each phase validates and builds on the previous one |

In [ ]:
crew = Crew(
    agents=agents,
    tasks=tasks,
    process=Process.sequential,
    verbose=True,
    memory=False,
)

print(f"Crew assembled: {len(crew.agents)} agents, {len(crew.tasks)} tasks")
print(f"Process: {crew.process}")

In [ ]:
# Execute the property analysis pipeline
print("=" * 70)
print(f"LAUNCHING REAL ESTATE ANALYSIS CREW — {datetime.now().strftime('%H:%M:%S')}")
print(f"Property: {PROPERTY['address']}")
print(f"Asking: ${PROPERTY['asking_price']:,} | Intent: {PROPERTY['intent']}")
print(f"Flow: Market -> Neighborhood -> Pricing -> Risk")
print("=" * 70)

result = crew.kickoff()

print("\n" + "=" * 70)
print(f"CREW COMPLETED — {datetime.now().strftime('%H:%M:%S')}")
print("=" * 70)

## 6. Analyze Results

### 6.1 Final Output — Risk Assessment & Recommendation

In [ ]:
print("=" * 70)
print("FINAL OUTPUT — Risk Assessment & Investment Recommendation")
print("=" * 70)
print(result.raw)

### 6.2 Individual Phase Outputs

Trace how information flowed through the analysis pipeline — from macro market conditions down to specific deal risk.

In [ ]:
for name, task in zip(task_names, tasks):
    print("\n" + "=" * 70)
    print(f"PHASE: {name}")
    print(f"Agent: {task.agent.role}")
    print("=" * 70)
    if task.output:
        text = task.output.raw
        if len(text) > 2500:
            print(text[:2500])
            print(f"\n... [{len(text) - 2500} more characters]")
        else:
            print(text)
    else:
        print("[No output captured]")

### 6.3 Pipeline Metrics

In [ ]:
print("PROPERTY ANALYSIS PIPELINE METRICS")
print("=" * 60)
print(f"{'Phase':<25} {'Agent':<30} {'Output':>8}")
print("-" * 60)

total = 0
for name, task in zip(task_names, tasks):
    length = len(task.output.raw) if task.output else 0
    total += length
    agent_short = task.agent.role.split("&")[0].strip()[:28]
    print(f"{name:<25} {agent_short:<30} {length:>6,}")

print("-" * 60)
print(f"{'TOTAL':<55} {total:>6,}")
print(f"\nProperty: {PROPERTY['address']}")
print(f"Asking Price: ${PROPERTY['asking_price']:,}")

if hasattr(result, 'token_usage') and result.token_usage:
    print(f"Tokens: {result.token_usage.get('total_tokens', 'N/A')}")

## 7. Save Analysis Report

In [ ]:
# Build and save the complete property analysis report
report_lines = [
    f"# Property Investment Analysis: {PROPERTY['address']}",
    f"**Generated:** {datetime.now().isoformat()}",
    f"**Property:** {PROPERTY['type']} | {PROPERTY['bedrooms']}bd/{PROPERTY['bathrooms']}ba | {PROPERTY['sqft']:,} sqft",
    f"**Asking Price:** ${PROPERTY['asking_price']:,}",
    f"**Intent:** {PROPERTY['intent']}",
    "---",
]

for name, task in zip(task_names, tasks):
    report_lines.append(f"\n## {name}\n")
    report_lines.append(task.output.raw if task.output else "[No output]")
    report_lines.append("\n---")

report = "\n".join(report_lines)
with open("property_analysis.md", "w", encoding="utf-8") as f:
    f.write(report)

print(f"Report saved: property_analysis.md ({len(report):,} chars)")

## 8. Experiment: Different Property & Market

Run the same crew on a completely different property to see how the agents adapt their analysis to different markets, property types, and investment theses.

In [ ]:
# =====================================================
# SECOND PROPERTY — Different market, different thesis
# =====================================================
PROPERTY_2 = {
    "address": "2215 Lake Shore Drive, Chicago, IL 60614",
    "type": "Condo (2-unit building)",
    "bedrooms": 2,
    "bathrooms": 1,
    "sqft": 1_100,
    "lot_sqft": 0,  # Condo
    "year_built": 1965,
    "asking_price": 320_000,
    "condition": "Fair — needs cosmetic updates, HVAC replaced 2021, original plumbing",
    "intent": "Value-add renovation then rent",
    "estimated_rent": 1_900,
}

print("PROPERTY 2 DETAILS")
print("=" * 50)
for k, v in PROPERTY_2.items():
    label = k.replace("_", " ").title()
    if isinstance(v, int) and v > 1000:
        print(f"  {label:<20} {v:>,}")
    else:
        print(f"  {label:<20} {v}")

In [ ]:
prop2_summary = (
    f"Property: {PROPERTY_2['address']}\n"
    f"Type: {PROPERTY_2['type']} | {PROPERTY_2['bedrooms']}bd/{PROPERTY_2['bathrooms']}ba | "
    f"{PROPERTY_2['sqft']:,} sqft\n"
    f"Year Built: {PROPERTY_2['year_built']} | Condition: {PROPERTY_2['condition']}\n"
    f"Asking Price: ${PROPERTY_2['asking_price']:,} | Est. Rent: ${PROPERTY_2['estimated_rent']:,}/mo\n"
    f"Intent: {PROPERTY_2['intent']}"
)

task2_market = Task(
    description=(
        f"Analyze market conditions for:\n{prop2_summary}\n\n"
        "Cover: market phase, supply/demand, price trends, interest rate impact, "
        "12-month forecast, and investment climate grade."
    ),
    expected_output="Market conditions report with phase, trends, forecast, and grade.",
    agent=market_analyst,
)

task2_neighborhood = Task(
    description=(
        f"Score the neighborhood for:\n{prop2_summary}\n\n"
        "Rate 10 dimensions (1-10), overall grade, neighborhood phase, top 3 "
        "strengths, top 3 concerns, and rental demand assessment."
    ),
    expected_output="Neighborhood scorecard with 10-dimension ratings and overall grade.",
    agent=neighborhood_scorer,
)

task2_pricing = Task(
    description=(
        f"Determine fair value and offer strategy for:\n{prop2_summary}\n\n"
        "Include comp analysis, FMV range, income valuation (GRM, NOI, cap rate), "
        "value-add renovation ROI estimate, and offer strategy."
    ),
    expected_output="Pricing analysis with FMV range, income metrics, and offer strategy.",
    agent=pricing_agent,
)

task2_risk = Task(
    description=(
        f"Risk assessment for:\n{prop2_summary}\n\n"
        "Evaluate 6 risk categories with likelihood/impact, include renovation risk "
        "(cost overruns, timeline, permits), stress test, exit assessment, "
        "and GO/CAUTION/NO-GO recommendation."
    ),
    expected_output="Risk report with 6-category analysis and investment recommendation.",
    agent=risk_reviewer,
)

tasks_2 = [task2_market, task2_neighborhood, task2_pricing, task2_risk]
print(f"Property 2 tasks created: {len(tasks_2)} tasks")

In [ ]:
crew2 = Crew(
    agents=agents,
    tasks=tasks_2,
    process=Process.sequential,
    verbose=True,
    memory=False,
)

print(f"Launching Crew 2 — {datetime.now().strftime('%H:%M:%S')}")
print(f"Property: {PROPERTY_2['address']}")
print("=" * 70)

result2 = crew2.kickoff()

print(f"\nCrew 2 completed — {datetime.now().strftime('%H:%M:%S')}")

In [ ]:
print("=" * 70)
print(f"RISK ASSESSMENT — {PROPERTY_2['address']}")
print("=" * 70)
print(result2.raw)

## 9. Compare Both Properties

In [ ]:
print("PROPERTY COMPARISON — PIPELINE OUTPUTS")
print("=" * 70)
print(f"{'Phase':<25} {'Austin SFH (chars)':<20} {'Chicago Condo (chars)':<20}")
print("-" * 70)

for name, t1, t2 in zip(task_names, tasks, tasks_2):
    len1 = len(t1.output.raw) if t1.output else 0
    len2 = len(t2.output.raw) if t2.output else 0
    print(f"{name:<25} {len1:<20,} {len2:<20,}")

total1 = sum(len(t.output.raw) for t in tasks if t.output)
total2 = sum(len(t.output.raw) for t in tasks_2 if t.output)
print("-" * 70)
print(f"{'TOTAL':<25} {total1:<20,} {total2:<20,}")

print(f"\nProperty 1: {PROPERTY['address']}")
print(f"  Asking: ${PROPERTY['asking_price']:,} | Type: {PROPERTY['type']}")
print(f"Property 2: {PROPERTY_2['address']}")
print(f"  Asking: ${PROPERTY_2['asking_price']:,} | Type: {PROPERTY_2['type']}")

## 10. Advanced: Explicit Context for Parallel Assessments

In some workflows, you want multiple agents to independently assess the same property and then have a final agent reconcile their views. This "parallel assessment" pattern avoids groupthink.

### Parallel Assessment Pattern

```
                   ┌──> Neighborhood Scorer ──┐
Market Analyst ────┤                          ├──> Risk Reviewer (reconciles)
                   └──> Pricing Agent ────────┘
```

The Neighborhood Scorer and Pricing Agent both receive the market analysis but NOT each other's output. The Risk Reviewer then sees all three and reconciles any conflicts.

In [ ]:
# Parallel assessment demo with explicit context
parallel_market = Task(
    description=(
        f"Market analysis for: {PROPERTY['address']}\n"
        f"{prop_summary}\n"
        "Cover market phase, supply/demand, price trends, and 12-month forecast."
    ),
    expected_output="Market conditions report.",
    agent=market_analyst,
)

# Both receive market context, but NOT each other's output
parallel_neighborhood = Task(
    description=(
        f"Neighborhood scoring for: {PROPERTY['address']}\n"
        f"{prop_summary}\n"
        "Rate 10 dimensions and provide overall grade."
    ),
    expected_output="Neighborhood scorecard.",
    agent=neighborhood_scorer,
    context=[parallel_market],  # Only sees market analysis
)

parallel_pricing = Task(
    description=(
        f"Pricing analysis for: {PROPERTY['address']}\n"
        f"{prop_summary}\n"
        "Determine FMV range and offer strategy using market data."
    ),
    expected_output="Pricing analysis with FMV range.",
    agent=pricing_agent,
    context=[parallel_market],  # Only sees market analysis (not neighborhood)
)

# Risk reviewer sees ALL three — reconciles independent assessments
parallel_risk = Task(
    description=(
        f"Risk review for: {PROPERTY['address']}\n"
        f"{prop_summary}\n"
        "You have received INDEPENDENT assessments from three agents: a market "
        "analyst, a neighborhood scorer, and a pricing agent. The neighborhood "
        "scorer and pricing agent analyzed independently (neither saw the other's "
        "output). Reconcile any conflicts between their assessments. Deliver a "
        "GO/CAUTION/NO-GO recommendation."
    ),
    expected_output="Reconciled risk review with GO/CAUTION/NO-GO.",
    agent=risk_reviewer,
    context=[parallel_market, parallel_neighborhood, parallel_pricing],
)

parallel_tasks = [parallel_market, parallel_neighborhood, parallel_pricing, parallel_risk]

print("PARALLEL ASSESSMENT FLOW")
print("=" * 60)
print("                    ┌──> Neighborhood Scorer ──┐")
print("  Market Analyst ───┤                          ├──> Risk Reviewer")
print("                    └──> Pricing Agent ────────┘")
print()
for i, task in enumerate(parallel_tasks, 1):
    ctx = [t.agent.role.split("&")[0].strip()[:25] for t in (task.context or [])]
    ctx_str = " + ".join(ctx) if ctx else "none (auto)"
    agent_short = task.agent.role.split("&")[0].strip()[:30]
    print(f"  Task {i}: {agent_short}")
    print(f"    Context from: {ctx_str}")

In [ ]:
# Run the parallel assessment crew
crew_parallel = Crew(
    agents=agents,
    tasks=parallel_tasks,
    process=Process.sequential,  # Sequential execution, but context controls data flow
    verbose=True,
    memory=False,
)

print(f"Launching parallel-assessment crew — {datetime.now().strftime('%H:%M:%S')}")
result_parallel = crew_parallel.kickoff()
print(f"\nParallel-assessment crew completed — {datetime.now().strftime('%H:%M:%S')}")

In [ ]:
# Show parallel assessment results
print("PARALLEL ASSESSMENT RESULTS")
print("=" * 60)
par_names = ["Market Analysis", "Neighborhood (independent)", "Pricing (independent)", "Risk (reconciled)"]
for name, task in zip(par_names, parallel_tasks):
    length = len(task.output.raw) if task.output else 0
    ctx = [t.agent.role.split("&")[0].strip()[:20] for t in (task.context or [])]
    ctx_str = ", ".join(ctx) if ctx else "auto"
    print(f"  {name:<30} {length:>6,} chars | Context: {ctx_str}")

## 11. Key Takeaways

### What We Built
- A **4-agent property investment analysis pipeline** (Market → Neighborhood → Pricing → Risk)
- Ran it on **two properties** (Austin SFH + Chicago condo) with different investment theses
- Demonstrated **parallel assessment** with explicit `context` to avoid groupthink

### Structural Patterns for Domain-Expert Crews
1. **Broad-to-Narrow scoping** — Start with macro context (market) and progressively narrow to property-specific analysis
2. **Constructive-then-Adversarial** — Build the investment thesis (market, neighborhood, pricing) then stress-test it (risk review)
3. **Structured output frameworks** — Scorecards (10-dimension), risk matrices (likelihood × impact), and deal grades (A-D) produce consistent, comparable output
4. **Parameterized pipelines** — A single property dict drives the entire analysis, making the crew reusable for any property

### Agent Design Lessons
- **Market Analyst:** Define a market phase framework (Recovery/Expansion/Hyper-Supply/Recession) so output is categorizable, not just descriptive
- **Neighborhood Scorer:** Use a structured scorecard with numerical ratings — it forces systematic evaluation and enables comparison across properties
- **Pricing Agent:** Mandate a specific methodology (comp analysis with adjustments) rather than allowing free-form price guessing
- **Risk Reviewer:** Make the adversarial mandate explicit in the backstory — "your job is to find what could go wrong, not validate the thesis"

### Production Tips
- Add real data tools: Zillow/Redfin API for comps, Census API for demographics, Walk Score API for walkability
- Use `output_pydantic` with a `PropertyAnalysis` schema for structured JSON output
- Enable `memory=True` to build up knowledge across multiple property analyses in the same market
- Consider adding a **Financial Modeler** agent that builds a detailed 10-year pro forma
- Add a **Legal/Compliance Agent** for markets with complex landlord-tenant regulations